# Givens rotations via a phase gradient

Bind real and complex Givens rotations to a shared gradient state and inspect
their registers, source call graphs, and emitted instructions.

Adapted from the Qualtran v0.7.0 Givens tutorial. Copyright 2023 Google LLC.
Licensed under Apache-2.0.

In [ ]:
from collections import Counter

from IPython.display import SVG, display

from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.algorithms.primitives.gradient_givens import (
    gradient_givens_cost, gradient_givens_on, gradient_givens_signature,
)
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources
from flagquantum.drawer.call_graph import export_modeled_call_graph_dot, render_call_graph_svg
from flagquantum.dynamic import DynamicCircuit

def show_signature(signature, model):
    display(SVG(field_signature_svg([(model.name, signature, {})])))

def show_calls(model):
    dot = export_modeled_call_graph_dot(model, max_depth=1)
    display(SVG(render_call_graph_svg(dot)))
    print("Declared resources:", estimate_component_resources(model))

## Real Givens rotation

Two Rz operations reuse the same angle and phase-gradient words, with a
Clifford ladder on the outside. Supply unsigned, big-endian fractional angle
bits and a negative-exponent phase-gradient state. An encoded fraction
x drives each Rz at angle 4πx. The target, angle, and gradient ports
are through ports. The temporary carries return to zero between rotations.

### Example

In [ ]:
width = 4
real_signature = gradient_givens_signature(width)
real_model = gradient_givens_cost(width)

### Register signature

In [ ]:
show_signature(real_signature, real_model)

### Call graph

In [ ]:
show_calls(real_model)

## Complex Givens rotation

The real rotation uses the real angle word. A separate Rz on the second
target uses the complex angle word. The adjoint applies the complex-angle
phase first and reverses both rotations.

### Example

In [ ]:
complex_signature = gradient_givens_signature(width, complex_rotation=True)
complex_model = gradient_givens_cost(width, complex_rotation=True)

### Register signature

In [ ]:
show_signature(complex_signature, complex_model)

### Call graph

In [ ]:
show_calls(complex_model)

### Bind and count the emitted instructions

The two target wires are followed by the real angle, complex angle, gradient,
and three clean carry wires. This example only builds the circuit and counts
its instructions. Prepare the gradient state before executing the rotation.
Classical slots 0–2 are reused after each carry cleanup.

In [ ]:
operation = gradient_givens_on(
    0, 1, (2, 3, 4, 5), (10, 11, 12, 13),
    complex_angle=(6, 7, 8, 9),
    workspace=(14, 15, 16),
    classical_bits=(0, 1, 2),
)
circuit = DynamicCircuit(17).append(operation)
emitted = Counter(gate.name for gate in circuit.to_ir().instructions)
print("Emitted T and T adjoints:", emitted["t"] + emitted["tdg"])
print("Emitted measurements and resets:", emitted["measure"], emitted["reset"])
assert emitted["t"] + emitted["tdg"] == 36
assert emitted["measure"] == emitted["reset"] == 9